<a href="https://colab.research.google.com/github/hindxb/FDS/blob/main/Notebooks/Project/FDS_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Arwa's Part - Feature Engineering**

#Feature Engineering
---
This phase includes
* Slight safety check after preprocessing
* Feature Engineering (mainly)
* Data splitting (train/test)



Step 1 - Define target output
---

The main objective based on the current data is to predict SPEED_FILLED

---
Based on the current data:
- road segment
- direction
- timestamp
- speed_filled (cleaned version of speed with no missing values)

the chosen target objective is most suitable as its directly correlated to traffic flow and congestion levels; **Predict the next traffic speed for each segment using previous time steps**. Thus, the model learns whether the traffic is staying stable, improving or worsening.

We will use multivariate time series prediction (multiple features) for better prediction


Step 2 - Prepare, Clean and Sort Data Correctly (Safety CHECK)
---
- Clean time order within each segment is required
- Same global start/end time across segements is not required, becuase the model is trying to learn patterns from each segment (uncombined) so its fine.

0. Load data
1. Parse time
2. Sort data
3. Filter segments with enough rows
4. Create temporal features (time-based features extraction)
5. Cyclical encoding for hour and day of week (feature transformation)
6. Keep only meaningful columns (features)
7. Convert to numeric columns
8. Handle missing values
9. Encode categorical features
10. Cleanup

In [ ]:
import numpy as np                 #for numerical operations such as sine and cosine
import pandas as pd                #for data loading and table dataframe manipulation
from google.colab import drive

#mount Google Drive
drive.mount('/content/drive')

#load file
file_path = '/content/drive/MyDrive/Datasets/chicago_traffic_imputed.csv'
df = pd.read_csv(file_path)        #read the CSV into a pandas data frame

print("Original shape:", df.shape) #print rows and cols in the original dataframe
print("\nOriginal columns:")
print(df.columns.tolist())         #convert cols names into Python list and print them


# 1. PARSE TIME
#==================================
df['TIME'] = pd.to_datetime(df['TIME'], dayfirst=True, errors='coerce')  #convert TIME to datetime assuming DD/MM/YY HH:mm format and set invalid values to NaT
# =========================================================
# CHECK ORIGINAL TIME VALUES BEFORE DROPPING
# =========================================================

# Save original TIME values if not already saved
if 'TIME_ORIGINAL' not in df.columns:
    df['TIME_ORIGINAL'] = df['TIME']

#Find rows where parsing failed
invalid_time_rows = df[df['TIME'].isna()]
print("\nNumber of invalid TIME rows:", len(invalid_time_rows))

df = df.dropna(subset=['TIME']).copy()                                   #remove rows where TIME couldn't be parsed, then create a copy of the cleaned dataframe

# 2. SORT DATA
#==================================
df = df.sort_values(['SEGMENT_ID', 'TIME']).reset_index(drop=True)       #sort rows by SEGMENT_ID then by TIME, then resent the index

# 3. FILTER SEGMENTS WITH ENOUGH ROWS
#We use a stronger threshold because sequence models
#need enough history per segment.
#==================================
min_rows_per_segment = 60          #defining min number of rows required per segment

segment_sizes = df.groupby('SEGMENT_ID').size()                                #count of rows per segment
valid_segments = segment_sizes[segment_sizes >= min_rows_per_segment].index    #keep only segment IDs whose rows meets the min threshold

df = df[df['SEGMENT_ID'].isin(valid_segments)].copy()                          #filter the dataframe to keep valid segments only then make a copy
df = df.sort_values(['SEGMENT_ID', 'TIME']).reset_index(drop=True)             #resort the filtered dataframe by segment and time, then reset index

print("\nShape after filtering segments:", df.shape)
print("Number of valid segments:", df['SEGMENT_ID'].nunique())                 #number of unique remaining segments


# 4. CREATE TEMPORAL FEATURES
#==================================
df['HOUR'] = df['TIME'].dt.hour                 #extract hour from TIME column
df['DAY_OF_WEEK'] = df['TIME'].dt.dayofweek     #extract day of week from TIME column; Mon=0, Sun=6

df['is_weekend'] = df['DAY_OF_WEEK'].isin([5, 6]).astype(int)                 #create a binary feature that is 1 for Sat or Sun, else 0
df['is_rush_hour'] = df['HOUR'].isin([7, 8, 9, 16, 17, 18]).astype(int)       #create a binary feature that is 1 during day time (rush hours), else 0

# 5. CYCLICAL ENCODING FOR HOUR AND DAY OF WEEK
#==============================================
df['hour_sin'] = np.sin(2 * np.pi * df['HOUR'] / 24)                          #encode hour cyclically using sin to preserve time of day circularity
df['hour_cos'] = np.cos(2 * np.pi * df['HOUR'] / 24)                          #encode hour cyclically using cosine

#cyclical encoding for day of week
df['dow_sin'] = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 7)                     #encode day of week cyclicaly using sin
df['dow_cos'] = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 7)                     #encode day of week cyclicaly using cos


# 6. KEEP ONLY MEANINGFUL COLUMNS
#Drop: redundant IDs, raw lat/lon, text streets,
#duplicate time, raw SPEED (leaky), raw hour/day of week/month
#====================================
candidate_cols = [
    'TIME',
    'SEGMENT_ID',

    #target
    'SPEED_FILLED',

    #traffic signals
    'BUS_COUNT',
    'MESSAGE_COUNT',
    'LENGTH',

    #missing indicators
    'SPEED_WAS_MISSING',

    #direction (will be one-hot encoded later)
    'DIRECTION',

    #temporal binary
    'is_weekend',
    'is_rush_hour',

    #temporal cyclical (already encoded so raw HOUR, DAY_OF_WEEK, MONTH ae dropped)
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
]

existing_cols = [col for col in candidate_cols if col in df.columns]    #keep only those candidate columns that actually exist in the data frame
df_model = df[existing_cols].copy()                                     #create a new dataframe containing only the selcted existing columns

print("\nShape after selecting columns:", df_model.shape)
print("Columns kept:")
print(df_model.columns.tolist())


# 7. CONVERT TO NUMERIC COLUMNS
#==================================
numeric_cols = [                    #list of cols expected to be numeric
    'SPEED_FILLED',
    'BUS_COUNT',
    'MESSAGE_COUNT',
    'LENGTH',
    'SPEED_WAS_MISSING',
    'is_weekend',
    'is_rush_hour',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos'
    'speed_lag_24',
    'segment_speed_mean',
    'month_sin',
    'month_cos',
]

for col in numeric_cols:
    if col in df_model.columns:     #check whether the current sol exists in df_model before converting
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')         #convert col to numeric and replace invalid values with Nan


# 8. HANDLE MISSING VALUES
#Add missing indicators before filling counts with 0.
#====================================
if 'BUS_COUNT' in df_model.columns:
    df_model['BUS_COUNT_MISSING'] = df_model['BUS_COUNT'].isna().astype(int)          #create a binary indicator (T/F) showing where BUS_COUNT was missing
    df_model['BUS_COUNT'] = df_model['BUS_COUNT'].fillna(0)                           #replace missing BUS_COUNT values with 0

if 'MESSAGE_COUNT' in df_model.columns:
    df_model['MESSAGE_COUNT_MISSING'] = df_model['MESSAGE_COUNT'].isna().astype(int)  #create a binary indicator (T/F) showing where MESSAGE_COUNT was missing
    df_model['MESSAGE_COUNT'] = df_model['MESSAGE_COUNT'].fillna(0)                   #replace missing MESSAGE_COUNT values with 0

if 'SPEED_WAS_MISSING' in df_model.columns:
    df_model['SPEED_WAS_MISSING'] = df_model['SPEED_WAS_MISSING'].fillna(0)           #this was imputed to cread SPEED_FILLED, whether it's important or not is discussed in the text just above this code block

#target must exist
df_model = df_model.dropna(subset=['SPEED_FILLED']).copy()                            #remove rows where target SPEED_FILLED is missing, then create a copy



# 9. ENCODE CATEGORICAL FEATURES
#====================================
categorical_cols = []                                                                 #initialize an empty list to store columns that need encoding

if 'DIRECTION' in df_model.columns:
    categorical_cols.append('DIRECTION')

for col in categorical_cols:
    df_model[col] = df_model[col].astype(str)                                         #convert the categorical col to string before one-hot encoding

if categorical_cols:                                                                  #check whether there is at least one categorical col to encode
    df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)    #one-hot encode categorical col and dropthe first category to reduce redundancy

# 10. CLEANUP
# We avoid aggressive cleanup on all columns unless needed
#=========================================================
required_cols = ['TIME', 'SEGMENT_ID', 'SPEED_FILLED']                                #defining cols that must not be missing
df_model = df_model.dropna(subset=required_cols).reset_index(drop=True)               #drop rows missing any required col and reset index

print("\nFinal shape after preprocessing:", df_model.shape)
print("\nFinal columns:")
print(df_model.columns.tolist())    #print final list of cols after all transformations

display(df_model.head())            #display the first few rows of the final preprocessed dataframe

Mounted at /content/drive


/tmp/ipykernel_30590/2606708916.py:10: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)        #read the CSV into a pandas data frame


Original shape: (517857, 25)

Original columns:
['TIME', 'SEGMENT_ID', 'SPEED', 'STREET', 'DIRECTION', 'FROM_STREET', 'TO_STREET', 'LENGTH', 'STREET_HEADING', 'COMMENTS', 'BUS_COUNT', 'MESSAGE_COUNT', 'HOUR', 'DAY_OF_WEEK', 'MONTH', 'RECORD_ID', 'START_LATITUDE', 'START_LONGITUDE', 'END_LATITUDE', 'END_LONGITUDE', 'START_LOCATION', 'END_LOCATION', 'TIME_ORIGINAL', 'SPEED_WAS_MISSING', 'SPEED_FILLED']

Number of invalid TIME rows: 0

Shape after filtering segments: (517857, 25)
Number of valid segments: 29

Shape after selecting columns: (517857, 14)
Columns kept:
['TIME', 'SEGMENT_ID', 'SPEED_FILLED', 'BUS_COUNT', 'MESSAGE_COUNT', 'LENGTH', 'SPEED_WAS_MISSING', 'DIRECTION', 'is_weekend', 'is_rush_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']

Final shape after preprocessing: (517856, 17)

Final columns:
['TIME', 'SEGMENT_ID', 'SPEED_FILLED', 'BUS_COUNT', 'MESSAGE_COUNT', 'LENGTH', 'SPEED_WAS_MISSING', 'is_weekend', 'is_rush_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'B

,TIME,SEGMENT_ID,SPEED_FILLED,BUS_COUNT,MESSAGE_COUNT,LENGTH,SPEED_WAS_MISSING,is_weekend,is_rush_hour,hour_sin,hour_cos,dow_sin,dow_cos,BUS_COUNT_MISSING,MESSAGE_COUNT_MISSING,DIRECTION_SB,DIRECTION_WB
0,2025-04-28 12:20:00,144,25.0,0,0,0.5,1.0,0,0,1.224647e-16,-1.000000,0.0,1.0,0,0,True,False
1,2025-04-28 12:30:00,144,25.5,0,0,0.5,1.0,0,0,1.224647e-16,-1.000000,0.0,1.0,0,0,True,False
2,2025-04-28 12:40:00,144,25.0,3,21,0.5,0.0,0,0,1.224647e-16,-1.000000,0.0,1.0,0,0,True,False
3,2025-04-28 12:50:00,144,25.5,0,0,0.5,1.0,0,0,1.224647e-16,-1.000000,0.0,1.0,0,0,True,False
4,2025-04-28 13:01:00,144,26.0,1,7,0.5,0.0,0,0,-2.588190e-01,-0.965926,0.0,1.0,0,0,True,False


Step 3 - Create Features and Remove NaN rows
---





We are taking traffic data and creating new features, to further help us understand what happened in the past, so we can predict what will happen in the future


---
11. Copy and sort
- Make a copy of data to preserve the original data.
- Arrange the data so that each road segment is grouped together
and inside each segment, time is ordered (past to future).
12. Create lag features
- By choosing few past time steps points' shifts from the past [1,2,3,6,12,24].
- For each road segment we take SPEED_FILLED column and shift it by each of these offsets, so that past speed values are copied into the current row as a new features.
- Loop through the list [1,2,3,6,12,24] to create columns
- Now we have new columns; speed_lag_1, speed_lag_2, speed_lag_3, speed_lag_6, etc.
- inside these columns we have shifted speeds from the past, where I got them from SPEED_FILL.
13. For each road segment, we take 80% of time and mark them as training data to avoid using future data, as we are trying to predict based on historical data
- For each road segment
- We take SPEED_FILLED values, but only from the first 80% of time
- We compute their average to get a new feature (segment_speed_mean) that is shared across all rows of the same segment.
14. Add seasonality (month features) because traffic may change based on the time of year by encoding months cyclically to ensure adjacent months are close to each other
15. Add rolling features to summarize recent short and long term traffic behavior
- By computing the average of the last 6 and 12 past values from SPEED_FILLED (within the same segment).
- By computing the standard deviation of the last 12 steps.
16. Add difference features reflecting the difference between current and past to mainly interpret speed increasing and decreasing (trend).
17. Collect all required features and remove incomplete rows.



In [ ]:
# 11. DATA PREPARATION AND TEMPORAL ORDERING
#===========================================
df = df_model.copy()                                                                    #create a copy so changes do not affect
df = df.sort_values(['SEGMENT_ID', 'TIME']).reset_index(drop=True)                      #sort the dataframe by segment and timestamp, then reset row numbering


# 12. SEGMENT-LEVEL STATISTICAL FEATURES: extended lags
#===========================================
for lag in [1, 2, 3, 6, 12, 24]:                                                        #loop through selected lag values to capture a longer historical pattern
    df[f'speed_lag_{lag}'] = df.groupby('SEGMENT_ID')['SPEED_FILLED'].shift(lag)        #for each segment, shift SPEED_FILLED downward by the lag amount to create a past-speed feature. shift(lag) means take the value from lag rows before

# 13. SEGEMNT SPEED MEAN: uses only training portion (first 80% per segment)
#===========================================
train_mask = df.groupby('SEGMENT_ID')['TIME'].transform(lambda x: x <= x.quantile(0.8))     #create a boolean mask of rows whose time is within the first 80% of timestamps inside each segment
segment_mean_train = df[train_mask].groupby('SEGMENT_ID')['SPEED_FILLED'].mean()            #compute the mean speed for eah segment using only rows marked as training
df['segment_speed_mean'] = df['SEGMENT_ID'].map(segment_mean_train)                         #map each segment ID to its training mean speed and store it as a new feature


# 14. TEMPORAL AND SEASONAL ENCODING
#features
#=========================================
df['month_sin'] = np.sin(2 * np.pi * df['TIME'].dt.month / 12)                              #encode month as a sin value to represent yearly cyclic seasonality
df['month_cos'] = np.cos(2 * np.pi * df['TIME'].dt.month / 12)                              #encode month as a cos value to complement the sin encoding


#15. ROLLING AND TREND BASED FEATURES
#=========================================
df['speed_roll_mean_6']  = df.groupby('SEGMENT_ID')['SPEED_FILLED'].transform(lambda x: x.shift(1).rolling(6,  min_periods=2).mean())     #for each segment, compute the rolling mean of the previous 6 speed values, using at least 2 past observations and excluding the current row with shift(1), rolling(6).mean(), means average over the last 6 past observations
df['speed_roll_std_6']   = df.groupby('SEGMENT_ID')['SPEED_FILLED'].transform(lambda x: x.shift(1).rolling(6,  min_periods=2).std())      #for each segment, compute the rolling standard deviation of the previous 6 speed values, uisng at least 2 past observations and excluding current row
df['speed_roll_mean_12'] = df.groupby('SEGMENT_ID')['SPEED_FILLED'].transform(lambda x: x.shift(1).rolling(12, min_periods=4).mean())     #for each segment, compute rolling mean of the previous 12 speed values, using at least 4 observations and excluding the current row

#16. SPEED DIFFERENCE FEATURE
#=========================================
df['speed_diff_1'] = df.groupby('SEGMENT_ID')['SPEED_FILLED'].diff(1)                       #for each segment, compute the difference between current and immediately previous speed. diff(1) means current value minus previous value
df['speed_diff_3'] = df.groupby('SEGMENT_ID')['SPEED_FILLED'].diff(3)                       #for each segment, compute the difference between the current and the speed value of 3 steps earlier

# 17. COLLECTION OF REQUIRED FEATURES
#drop rows with missing lag values
#=========================================
lag_cols = [f'speed_lag_{l}' for l in [1,2,3,6,12,24]] + \
           ['speed_roll_mean_6','speed_roll_std_6','speed_roll_mean_12',
            'speed_diff_1','speed_diff_3']                                                  #create a list for all lag feature names, add rolling statistic featuresto the list of required historical features, add speed difference features too

#data cleaning for model input
df = df.dropna(subset=lag_cols).reset_index(drop=True)                                      #remove rows where any required lag/rolling/difference feature is missing, then reset index. removes the first wors in each segment becasue they dont yet have enough history
print("Shape after lag/rolling features:", df.shape)                                        #dataframe shape after creating features and dropping incomplete rows

Shape after lag/rolling features: (517160, 31)


Step 4 - Feature Type Validation and Data Cleaning
---
18. Final numeric check
19. Final cleanup

In [ ]:
#initial checking
print("\nUsing dataframe after short-term lag creation")
print("Current shape:", df.shape)
print("Columns available:")
print(df.columns.tolist())    #print all the names of all columns in the dataframe as a list


# 18. FINAL NUMERIC CHECK
#Make sure all model columns except TIME (datetime object) are numeric where needed.
#=========================================================
numeric_cols = [
    'SPEED_FILLED',
    'BUS_COUNT',
    'MESSAGE_COUNT',
    'LENGTH',
    'SPEED_WAS_MISSING',
    'is_weekend',
    'is_rush_hour',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'BUS_COUNT_MISSING',
    'MESSAGE_COUNT_MISSING',

    #short-term speed lags
    'speed_lag_1',            #these are already created so they must be numeric for sure, adding or removing them won't hurt
    'speed_lag_2',
    'speed_lag_3'
]

for col in numeric_cols:
    if col in df.columns:     #check whether the current column actually exists in the dataframe
        df[col] = pd.to_numeric(df[col], errors='coerce')  #convert the column to numeric and replace invalid values with NaN

#also make sure one-hot encoded direction columns are numeric
dummy_cols = [col for col in df.columns if col.startswith('DIRECTION_')]    #find all columns related to direction
for col in dummy_cols:                                                      #for each of these columns
    df[col] = pd.to_numeric(df[col], errors='coerce')                       #convert each direction to numeric (0/1)

print("\nNumeric conversion check done")
print("Current shape:", df.shape)       #the dataframe shape after numeric conversion


# 19. FINAL CLEANUP
#========================================
before = df.shape[0]                    #store the number of rows before cleanup, so the removed count can be reported later

required_cols_for_model = [             #list of columns that must exist and must not be missing a row to be usable for the models
    'TIME',
    'SEGMENT_ID',
    'SPEED_FILLED',
    'speed_lag_1',
    'speed_lag_2',
    'speed_lag_3'
]

df = df.dropna(subset=required_cols_for_model).reset_index(drop=True)             #remove rows missing any required columns and reset indexing

after = df.shape[0]

print("\nRows before final cleanup:", before)
print("Rows after final cleanup :", after)
print("Rows removed             :", before - after)

print("\nFinal columns:")
print(df.columns.tolist())              #final list of column names after clean up

display(df.head())                      #display the first few rows of the final cleaned dataframe


Using dataframe after short-term lag creation
Current shape: (517160, 31)
Columns available:
['TIME', 'SEGMENT_ID', 'SPEED_FILLED', 'BUS_COUNT', 'MESSAGE_COUNT', 'LENGTH', 'SPEED_WAS_MISSING', 'is_weekend', 'is_rush_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'BUS_COUNT_MISSING', 'MESSAGE_COUNT_MISSING', 'DIRECTION_SB', 'DIRECTION_WB', 'speed_lag_1', 'speed_lag_2', 'speed_lag_3', 'speed_lag_6', 'speed_lag_12', 'speed_lag_24', 'segment_speed_mean', 'month_sin', 'month_cos', 'speed_roll_mean_6', 'speed_roll_std_6', 'speed_roll_mean_12', 'speed_diff_1', 'speed_diff_3']

Numeric conversion check done
Current shape: (517160, 31)

Rows before final cleanup: 517160
Rows after final cleanup : 517160
Rows removed             : 0

Final columns:
['TIME', 'SEGMENT_ID', 'SPEED_FILLED', 'BUS_COUNT', 'MESSAGE_COUNT', 'LENGTH', 'SPEED_WAS_MISSING', 'is_weekend', 'is_rush_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'BUS_COUNT_MISSING', 'MESSAGE_COUNT_MISSING', 'DIRECTION_SB', 'DIREC

,TIME,SEGMENT_ID,SPEED_FILLED,BUS_COUNT,MESSAGE_COUNT,LENGTH,SPEED_WAS_MISSING,is_weekend,is_rush_hour,hour_sin,...,speed_lag_12,speed_lag_24,segment_speed_mean,month_sin,month_cos,speed_roll_mean_6,speed_roll_std_6,speed_roll_mean_12,speed_diff_1,speed_diff_3
0,2025-04-28 16:20:00,144,20.5,0,0,0.5,1.0,0,1,-0.866025,...,28.0,25.0,24.835616,0.866025,-0.5,20.750000,1.224745,20.916667,-1.5,0.75
1,2025-04-28 16:30:00,144,19.0,3,19,0.5,0.0,0,1,-0.866025,...,22.0,25.5,24.835616,0.866025,-0.5,20.500000,1.060660,20.291667,-1.5,0.00
2,2025-04-28 16:40:00,144,20.0,2,10,0.5,0.0,0,1,-0.866025,...,16.0,25.0,24.835616,0.866025,-0.5,20.125000,1.137431,20.041667,1.0,-2.00
3,2025-04-28 16:50:00,144,17.0,4,20,0.5,0.0,0,1,-0.866025,...,19.5,25.5,24.835616,0.866025,-0.5,20.041667,1.122683,20.375000,-3.0,-3.50
4,2025-04-28 17:01:00,144,26.0,1,4,0.5,0.0,0,1,-0.965926,...,23.0,26.0,24.835616,0.866025,-0.5,19.583333,1.685724,20.166667,9.0,7.00


Step 5 - Training/Testing Splits Per Segment and Scaling
---

- Sort by time then take first 80% of each segment (training data) and last 20% (testing data). This way we ensure that we train on past data and test on future data.
- Scaling comes after splitting to avoid using test information causing cheating in model

In [ ]:
# 20. TRAIN / TEST SPLIT PER SEGMENT
#pandas and numpy were imported before,
#but still added them again for clarity
#======================================
from sklearn.preprocessing import MinMaxScaler      #for scaling values between 0 and 1
import numpy as np                                  #for numerical operations
import pandas as pd                                 #for dataframe operations

sequence_length = 24                                #define sequence length used later for sequence creation
split_ratio = 0.8                                   #set 80% of each segment for training

train_parts = []                                    #empty list to later store training parts from each segment
test_parts = []                                     #empty list to later store testing parts from each segment

for segment in df['SEGMENT_ID'].unique():           #loop through each unique segment
    segment_df = df[df['SEGMENT_ID'] == segment].sort_values('TIME').copy()   #select a segment and sort by time
    n = len(segment_df)                             #get count of rows in this segment

    split_idx = int(n * split_ratio)                #compute row index where trainign ends and testing begins

    #require enough rows for both train and test after sequence creation
    if split_idx <= sequence_length or (n - split_idx) <= sequence_length:    #skip segments that are too short for sequences
        continue

    train_parts.append(segment_df.iloc[:split_idx].copy())  #add 1st 80% of this segment to training data
    test_parts.append(segment_df.iloc[split_idx:].copy())   #add the last 20% of this segment to testing data

train_df = pd.concat(train_parts, axis=0).reset_index(drop=True)    #combine all segment training parts itno one dataframe
test_df = pd.concat(test_parts, axis=0).reset_index(drop=True)      #combine all segment testing parts into one dataframe

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train segments:", train_df['SEGMENT_ID'].nunique())
print("Test segments :", test_df['SEGMENT_ID'].nunique())           #number of training and tetsing segments should be same because I am taking percentage of rows from each segment then combine all rows based on whether training or testing


# 21. DEFINE FEATURE COLUMNS
#Keep only the real predictor features.
#No TIME, SEGMENT_ID, or target as normal predictors
#==========================================================
feature_cols = [col for col in df.columns if col not in ['TIME', 'SEGMENT_ID', 'SPEED_FILLED']]

print("\nNumber of predictor features:", len(feature_cols))
print("Predictor columns:")
print(feature_cols)

# 22. SCALE FEATURES AND TARGET
#Fit scalers on training data only
#==========================================================
feature_scaler = MinMaxScaler()     #create a scaler object for input features
target_scaler = MinMaxScaler()      #create a seperate scaler object for the target column

train_df[feature_cols] = feature_scaler.fit_transform(train_df[feature_cols])     #fit is to learn min and max from features and transform is to scale them to 0-1, then replace original values with scaled ones
test_df[feature_cols] = feature_scaler.transform(test_df[feature_cols])           #scale using min and max from training and reuse them for testing, then scale testing based on them, because tesing must behave like unseen future data

train_df[['SPEED_FILLED']] = target_scaler.fit_transform(train_df[['SPEED_FILLED']])    #learn min and max of speed from training and scale them
test_df[['SPEED_FILLED']] = target_scaler.transform(test_df[['SPEED_FILLED']])          #use same min and max from training and scale

print("\nScaling done")

Train shape: (413711, 31)
Test shape : (103449, 31)
Train segments: 29
Test segments : 29

Number of predictor features: 28
Predictor columns:
['BUS_COUNT', 'MESSAGE_COUNT', 'LENGTH', 'SPEED_WAS_MISSING', 'is_weekend', 'is_rush_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'BUS_COUNT_MISSING', 'MESSAGE_COUNT_MISSING', 'DIRECTION_SB', 'DIRECTION_WB', 'speed_lag_1', 'speed_lag_2', 'speed_lag_3', 'speed_lag_6', 'speed_lag_12', 'speed_lag_24', 'segment_speed_mean', 'month_sin', 'month_cos', 'speed_roll_mean_6', 'speed_roll_std_6', 'speed_roll_mean_12', 'speed_diff_1', 'speed_diff_3']

Scaling done
